In [1]:
!pip install transformers torch

In [2]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

def rqg_score(context, question, weak_th=0.4, strong_th=0.7):
    result = qa_pipeline(question=question, context=context)

    answer = result['answer'].strip()
    confidence = float(result['score'])

    invalid_answers = ["", " ", ".", ",", "the", "a", "an"]
    if answer.lower() in invalid_answers or len(answer) < 3:
        relevance = "Irrelevant"
        quality = "Very Weak"

    else:

        if confidence >= strong_th:
            relevance = "Relevant"
            quality = "Strong"
        elif confidence >= weak_th:
            relevance = "Partially Relevant"
            quality = "Moderate"
        else:
            relevance = "Weakly Relevant"
            quality = "Weak"

    return {
        "question": question,
        "predicted_answer": answer,
        "confidence": round(confidence, 4),
        "relevance": relevance,
        "quality": quality
    }

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [4]:
abstract = """
acquisition of semantic classes for adjectives from distributional evidence
in this paper , we present a clustering experiment directed at the acquisition of semantic classes for adjectives in catalan , using only shallow distributional features . we define a broad-coverage classification for adjectives based on ontological semantics . we classify along two parameters ( number of arguments and ontological kind of denotation ) , achieving reliable agreement results among human judges . the clustering procedure achieves a comparable agreement score for one of the parameters , and a little lower for the other.
"""

question = "What parameters are used for classification?"

output = rqg_score(abstract, question)

print(output)

{'question': 'What parameters are used for classification?', 'predicted_answer': 'number of arguments and ontological kind of denotation', 'confidence': 0.9538, 'relevance': 'Relevant', 'quality': 'Strong'}


In [5]:
!pip install transformers torch pandas tqdm

In [6]:
from google.colab import files

uploaded = files.upload()

Saving Claude AI Cofidence score - Sheet1.csv to Claude AI Cofidence score - Sheet1.csv


In [7]:
import pandas as pd

df = pd.read_csv(list(uploaded.keys())[0])

print(df.head())

                                   selected_abstract  \
0  acquisition of semantic classes for adjectives...   
1  acquisition of semantic classes for adjectives...   
2  acquisition of semantic classes for adjectives...   
3  acquisition of semantic classes for adjectives...   
4  "acquisition of semantic classes for adjective...   

                                             Human 1  \
0  We’re trying to group adjectivs into meaningfu...   
1  We’re trying to group adjectivs into meaningfu...   
2  We’re trying to group adjectivs into meaningfu...   
3  We’re trying to group adjectivs into meaningfu...   
4  We’re trying to group adjectivs into meaningfu...   

                            Questions from Claude AI  
0  What specific agreement metrics (e.g., kappa) ...  
1  How should the informal, conversational langua...  
2  How does the proposed clustering approach rela...  
3  What exact shallow distributional features wer...  
4  What ontological framework underlies the two-p..

In [8]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/deberta-v3-large-squad2"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

DebertaV2ForQuestionAnswering LOAD REPORT from: deepset/deberta-v3-large-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [9]:
import re

def split_questions(q_string):
    questions = re.split(r'\?\s+', str(q_string))
    return [q.strip() + '?' for q in questions if q.strip()]

def rqg_score(context, question, weak_th=0.15, strong_th=0.5):
    result = qa_pipeline(question=question, context=context)

    answer = result['answer'].strip()
    confidence = float(result['score'])

    if confidence >= strong_th:
        relevance = "Relevant"
        quality = "Strong"
    elif confidence >= weak_th:
        relevance = "Partially Relevant"
        quality = "Moderate"
    else:
        relevance = "Weakly Relevant"
        quality = "Weak"

    return answer, round(confidence, 4), relevance, quality

In [13]:
from tqdm import tqdm

results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    context = str(row["selected_abstract"]).replace("\n", " ")
    questions = split_questions(row["Questions from Claude AI"])

    for q in questions:
        answer, conf, rel, qual = rqg_score(context, q)

        results.append({
            "question": q,
            "predicted_answer": answer,
            "rqg_confidence": conf,
            "relevance": rel,
            "quality": qual
        })

100%|██████████| 500/500 [00:47<00:00, 10.58it/s]


In [15]:
output_df = pd.DataFrame(results)

output_file = "rqg_evaluation_output.csv"
output_df.to_csv(output_file, index=False)


In [16]:
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>